In [3]:
import numpy as np
import torch
import torch.nn as nn

np.random.seed(42)

# ── Setup: one linear layer + ReLU + MSE loss ────────────
n_in, n_out = 4, 3
x_np = np.random.randn(n_in).astype(np.float32)
W_np = np.random.randn(n_out, n_in).astype(np.float32)
b_np = np.random.randn(n_out).astype(np.float32)
t_np = np.random.randn(n_out).astype(np.float32)   # target

# ── MANUAL FORWARD PASS ──────────────────────────────────
z = W_np @ x_np + b_np          # linear: (3,)
a = np.maximum(0, z)             # ReLU:   (3,)
diff = a - t_np                  # residual
L = np.sum(diff**2) / n_out      # MSE loss: scalar

print("=== MANUAL FORWARD ===")
print(f"z     = {z.round(4)}")
print(f"a     = {a.round(4)}")
print(f"Loss  = {L:.6f}")

# ── MANUAL BACKWARD PASS ─────────────────────────────────
# Step 1: dL/da  (gradient of MSE loss)
dL_da = 2 * diff / n_out                     # (3,)

# Step 2: dL/dz  (through ReLU)
relu_mask = (z > 0).astype(np.float32)       # 1 where active, 0 where dead
dL_dz = dL_da * relu_mask                    # (3,)

# Step 3: dL/dW  (outer product)
dL_dW = np.outer(dL_dz, x_np)               # (3,4) = same shape as W ✓

# Step 4: dL/db  (bias gradient)
dL_db = dL_dz                                # (3,)

# Step 5: dL/dx  (input gradient — uses Wᵀ)
dL_dx = W_np.T @ dL_dz                      # (4,) = same shape as x ✓

print("=== MANUAL GRADIENTS ===")
print(f"dL/dW ={dL_dW.round(6)}")
print(f"dL/db = {dL_db.round(6)}")
print(f"dL/dx = {dL_dx.round(6)}")

# ── PYTORCH AUTOGRAD (ground truth) ──────────────────────
x_t = torch.tensor(x_np, requires_grad=True)
W_t = torch.tensor(W_np, requires_grad=True)
b_t = torch.tensor(b_np, requires_grad=True)
t_t = torch.tensor(t_np)

z_t = W_t @ x_t + b_t
a_t = torch.relu(z_t)
L_t = torch.mean((a_t - t_t)**2)

L_t.backward()   # ← PyTorch runs the chain rule on the computational graph

print("=== PYTORCH GRADIENTS ===")
print(f"dL/dW ={W_t.grad.numpy().round(6)}")
print(f"dL/db = {b_t.grad.numpy().round(6)}")
print(f"dL/dx = {x_t.grad.numpy().round(6)}")

# ── VERIFICATION — this is the moment of truth ───────────
print("=== VERIFICATION ===")
print(f"dL/dW match: {np.allclose(dL_dW, W_t.grad.numpy(), atol=1e-5)}")
print(f"dL/db match: {np.allclose(dL_db, b_t.grad.numpy(), atol=1e-5)}")
print(f"dL/dx match: {np.allclose(dL_dx, x_t.grad.numpy(), atol=1e-5)}")

# ── 2-LAYER NETWORK — extend to see chaining in action ───
print("=== 2-LAYER BACKPROP ===")
n1, n2, n3 = 4, 5, 2

W1 = np.random.randn(n2, n1).astype(np.float32)
b1 = np.random.randn(n2).astype(np.float32)
W2 = np.random.randn(n3, n2).astype(np.float32)
b2 = np.random.randn(n3).astype(np.float32)
x2 = np.random.randn(n1).astype(np.float32)
t2 = np.random.randn(n3).astype(np.float32)

# Forward
z1 = W1 @ x2 + b1
a1 = np.maximum(0, z1)
z2 = W2 @ a1 + b2
a2 = np.maximum(0, z2)
loss2 = np.mean((a2 - t2)**2)

# Backward — layer 2
dL_da2 = 2*(a2 - t2)/n3
dL_dz2 = dL_da2 * (z2 > 0)
dL_dW2 = np.outer(dL_dz2, a1)
dL_db2 = dL_dz2
dL_da1 = W2.T @ dL_dz2        # ← gradient flows back through W2ᵀ

# Backward — layer 1
dL_dz1 = dL_da1 * (z1 > 0)   # ← gradient flows through ReLU mask
dL_dW1 = np.outer(dL_dz1, x2)
dL_db1 = dL_dz1

# Verify with PyTorch
W1t=torch.tensor(W1,requires_grad=True); b1t=torch.tensor(b1,requires_grad=True)
W2t=torch.tensor(W2,requires_grad=True); b2t=torch.tensor(b2,requires_grad=True)
x2t=torch.tensor(x2,requires_grad=True)
a1t=torch.relu(W1t@x2t+b1t); a2t=torch.relu(W2t@a1t+b2t)
torch.mean((a2t-torch.tensor(t2))**2).backward()

print(f"Layer 1 dW match: {np.allclose(dL_dW1, W1t.grad.numpy(), atol=1e-5)}")
print(f"Layer 2 dW match: {np.allclose(dL_dW2, W2t.grad.numpy(), atol=1e-5)}")
print("All True = you have correctly implemented backpropagation from scratch.")
print("Push to GitHub: git add . && git commit -m 'Day 08: backprop from scratch' && git push")

=== MANUAL FORWARD ===
z     = [ 1.0949 -1.0034 -2.4969]
a     = [1.0949 0.     0.    ]
Loss  = 2.828388
=== MANUAL GRADIENTS ===
dL/dW =[[ 0.830242 -0.231104  1.082591  2.545696]
 [-0.        0.       -0.       -0.      ]
 [ 0.       -0.        0.        0.      ]]
dL/db = [ 1.671468 -0.        0.      ]
dL/dx = [-0.39138  -0.391352  2.639604  1.282743]
=== PYTORCH GRADIENTS ===
dL/dW =[[ 0.830242 -0.231104  1.082591  2.545696]
 [ 0.       -0.        0.        0.      ]
 [ 0.       -0.        0.        0.      ]]
dL/db = [1.671468 0.       0.      ]
dL/dx = [-0.39138  -0.391352  2.639604  1.282743]
=== VERIFICATION ===
dL/dW match: True
dL/db match: True
dL/dx match: True
=== 2-LAYER BACKPROP ===
Layer 1 dW match: True
Layer 2 dW match: True
All True = you have correctly implemented backpropagation from scratch.
Push to GitHub: git add . && git commit -m 'Day 08: backprop from scratch' && git push
